<div dir="rtl">
<h1>گزینهٔ عبورکننده از آستانه را گم نکنید</h1>
<p>درس 63 از 76 · چطور نامزدهای نامحتمل را کنار بگذاریم؟ · <code dir="ltr">56-topkp</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/56-topkp.html">📖 بازگشت به همین درس</a></p>
<p>ماسک Top-p را بنویسید و رفتار Top-k در امتیازهای مساوی را تعمیر کنید.</p><p>پیش‌نیاز: مرتب‌سازی، مجموع تجمعی و بازگرداندن اندیس‌ها.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>برای [0.6,0.25,0.1,0.05] و p=0.8 چند گزینه لازم است؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.sampling import filter_logits
torch.set_num_threads(1)
probabilities = torch.tensor([0.6,0.25,0.1,0.05])
print('project candidates:',torch.isfinite(filter_logits(probabilities.log()[None],top_p=0.8)).tolist())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>nucleus_mask(probabilities, p) برای یک بردار احتمال مثبت با مجموع یک و 0<p<=1، ماسک bool هم‌اندازه در ترتیب اصلی بدهد. کوچک‌ترین پیشوند مرتب با مجموع حداقل p حفظ شود؛ در تساوی احتمال، ترتیب اصلی حفظ شود.</p>
</div>

In [ ]:
def nucleus_mask(probabilities, p):
    # TODO: مرتب‌سازی، عبور از آستانه، بازگشت به ترتیب اصلی
    return None

In [ ]:
def test_exercise():
    result = nucleus_mask(probabilities,0.8)
    if result is None:
        return False
    assert result.dtype==torch.bool and result.tolist()==[True,True,False,False]
    assert nucleus_mask(probabilities,0.01).tolist()==[True,False,False,False]
    assert nucleus_mask(torch.tensor([0.5,0.5]),0.5).tolist()==[True,False]
    assert nucleus_mask(torch.tensor([0.1,0.6,0.05,0.25]),0.8).tolist()==[False,True,False,True]
    assert nucleus_mask(probabilities,1.0).all()
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: nucleus_mask')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط p را تغییر دهید و تعداد نامزدها را با پیاده‌سازی واقعی اندازه بگیرید؛ این تعداد ثابت نیست.</p>
</div>

In [ ]:
for p in (0.1,0.5,0.8,0.95,1.0):
    mask = torch.isfinite(filter_logits(probabilities.log()[None],top_p=p))
    print(p,mask.tolist(),int(mask.sum()))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>حذف امتیازهای کمتر از kاُم، هنگام تساوی بیش از k گزینه نگه می‌دارد. exact_topk_mask(Logits,k) برای بردار یک‌بعدی و 1<=k<=V ماسکی با دقیقاً k خانهٔ True بدهد؛ هویت گزینه‌های مساوی در این تمرین مهم نیست.</p>
</div>

In [ ]:
ties = torch.tensor([2.0,2.0,2.0,0.0])
wrong = ties>=ties.topk(2).values[-1]
print('wanted 2, kept:',int(wrong.sum()),wrong.tolist())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def exact_topk_mask(logits, k):
    # TODO: از اندیس گزینه‌ها استفاده کنید
    return None

In [ ]:
def test_repair():
    result = exact_topk_mask(torch.tensor([2.0,2.0,2.0,0.0]),2)
    if result is None:
        return False
    assert result.dtype==torch.bool and result.sum().item()==2
    assert not result[-1]
    assert exact_topk_mask(torch.tensor([0.0,3.0,1.0]),1).tolist()==[False,True,False]
    assert exact_topk_mask(torch.zeros(4),4).all()
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: exact_topk_mask')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>filter_logits در sampling.py با همین دو خطای مرزی روبه‌روست. در ترکیب دو روش، ابتدا Top-k اعمال و سپس Top-p روی توزیع باقی‌مانده حساب می‌شود.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا ماسک درست باید هم احتمال تجمعی را رعایت کند و هم ترتیب اصلی Vocabulary را نگه دارد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/56-topkp.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/56-topkp.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>